# CLIP ViT-B/16 — reproducible selective INT8/FP16 TensorRT pipeline

**Platform:** Google Colab GPU (Tesla T4 tested), Linux, CUDA.

The first cell checks the environment and installs dependencies. The second cell mounts Google Drive and runs all verified stages. Re-running the second cell reuses checksum-verified artifacts.

The published results came from a particular Tesla T4 runtime; exact timing and bitwise-identical TensorRT engine builds across machines are **not** guaranteed. Dataset access and free Colab GPU availability are external dependencies.

In [ ]:
import sys, subprocess, os
from pathlib import Path
assert Path('/content').exists(), 'Run in Google Colab with a GPU runtime.'
assert subprocess.run(['nvidia-smi'],capture_output=True).returncode == 0, 'Enable a GPU runtime (T4 recommended).'
# This cell assumes the repository is already cloned to /content/clip-tensorrt-cpp-inference
REPO = Path('/content/clip-tensorrt-cpp-inference')
assert (REPO / 'requirements.txt').exists(), 'Clone the repo before running. See README.'
os.environ['HF_HOME'] = '/content/hf'
os.environ['HF_HUB_CACHE'] = '/content/hf/hub'
subprocess.run([sys.executable, '-m','pip','install','-q','-r',str(REPO/'requirements.txt')],check=True)
print('Dependencies installed. If Torch/Transformers were already imported, restart the runtime once.')

## Execute the restartable stages

The Drive folder is persistent. Each large artifact is created under `/content` and copied to Drive with SHA-256 validation. You may run a single stage, for example `--stage build`, after dependencies are available.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys
root='/content/drive/MyDrive/CLIP_Quantization_Project'
subprocess.run([sys.executable,'-m','src.pipeline','--root',root,'--stage','all'], cwd=str(REPO),check=True)

## Saved results

The runner writes evaluation metrics, predictions, SHA-256 records, and 3 repeated GPU benchmarks for each batch/model pairing to `CLIP_Quantization_Project/results/`. Source scripts in `src/` are the authoritative implementation.

In [ ]:
from pathlib import Path
import pandas as pd
root=Path('/content/drive/MyDrive/CLIP_Quantization_Project')
for name in ['clip_baseline_fp16_metrics.json','clip_attention_fc1_int8_fp16_repaired_metrics.json']:
    file=root/'results'/name
    if file.exists(): print(name, file.read_text())
file=root/'results/final_trt_comparison.csv'
if file.exists(): display(pd.read_csv(file))